# The ablation, under the leakage-free protocol

Every published ablation number came from `08_train_colab.ipynb`, which runs the
**older protocol** — features computed over the whole graph, so a training-window
feature could contain information from the future. Those numbers are inflated and
cannot be compared with the served model, which was trained temporally.

This notebook retrains every stage the same way the served model was trained, so the
ablation and the deployed system are finally measured on the same footing.

| stage | what it adds |
|---|---|
| 1  | BaselineGraphSAGE, BCE + pos_weight |
| 2  | + Edge-MLP attention (Novelty 1) |
| 3a | + Focal loss (Novelty 2, loss half) |
| 3b | + Graph-aware sampler (Novelty 2, sampling half) — **this is what ships** |
| 3c | 3b **minus** the Edge-MLP — the leave-one-out arm for Novelty 1 |

**3c is the one that matters most.** Stage 2 vs 1 measures edge attention in
isolation, where it shows almost nothing. 3b vs 3c measures it inside the full
system, which is the only comparison that supports the claim.

Runs are **resumable**: a stage whose report is already on Drive is skipped, so a
Colab disconnect costs you one stage, not the whole run.


## 1 · Environment


In [ ]:
import os, pathlib, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

DRIVE  = pathlib.Path('/content/drive/MyDrive/deepsentinel')
RUNS   = DRIVE / 'temporal_ablation'      # results land here, survive a disconnect
RUNS.mkdir(parents=True, exist_ok=True)

REPO = pathlib.Path('/content/Graphsage')
if not REPO.exists():
    !git clone -q https://github.com/R26-IT-121/Graphsage.git {REPO}
%cd {REPO}
!git pull -q
!pip -q install -e . 2>&1 | tail -2

import torch
print('device:', 'cuda' if torch.cuda.is_available() else 'CPU  <-- check Runtime > Change runtime type')


## 2 · The feature table

`features.parquet` (65 MB) is the only input. Kept on Drive so it is built once.


In [ ]:
proc = REPO / 'data' / 'processed'; proc.mkdir(parents=True, exist_ok=True)
src = DRIVE / 'features.parquet'
if not (proc / 'features.parquet').exists():
    if src.exists():
        !cp {src} {proc}/features.parquet
    else:
        !python scripts/download_paysim.py && python scripts/prepare_features.py
        !cp {proc}/features.parquet {src}
print('features.parquet:', (proc / 'features.parquet').stat().st_size / 1e6, 'MB')


## 3 · The temporal graph

`--features v2` gives the 12 behavioural node features the served model uses.
Cached to Drive — building it takes a while and never changes.


In [ ]:
graph = REPO / 'data' / 'graph' / 'paysim_temporal_v2.pt'
graph.parent.mkdir(parents=True, exist_ok=True)
cached = DRIVE / 'paysim_temporal_v2.pt'

if cached.exists():
    !cp {cached} {graph}
else:
    !python scripts/build_temporal_graph.py --features v2
    !cp {graph} {cached}
print('graph:', graph.stat().st_size / 1e6, 'MB')


## 4 · Train every stage

Seed 0 first — that alone gives you the ablation table. Extra seeds come next,
and are what turn 'Novelty 1 helps' into a claim with a confidence interval.

Each stage copies its report, scores and checkpoint to Drive the moment it
finishes, so nothing is lost if the session drops.


In [ ]:
STAGES = ['1', '2', '3a', '3b', '3c']
SEEDS  = [0, 1, 2]    # seed 0 is done; these two turn it into a claim
EPOCHS = 50

def done(stage, seed):
    return (RUNS / f'stage{stage}_v2_seed{seed}.json').exists()

for seed in SEEDS:
    for stage in STAGES:
        if done(stage, seed):
            print(f'skip  stage {stage} seed {seed} — already on Drive'); continue
        print(f'\n=== stage {stage}, seed {seed} ' + '='*40)
        rc = subprocess.call([sys.executable, 'scripts/train_temporal.py',
                              '--stage', stage, '--seed', str(seed),
                              '--features', 'v2', '--epochs', str(EPOCHS)])
        if rc != 0:
            print(f'stage {stage} seed {seed} FAILED (exit {rc}) — continuing'); continue
        tag = f'stage{stage}_v2_seed{seed}'
        for pat in (f'reports/temporal/{tag}.json',
                    f'reports/temporal/{tag}_scores.pt',
                    f'checkpoints/temporal_{tag}.pt'):
            p = REPO / pat
            if p.exists():
                !cp {p} {RUNS}/
        print('saved to Drive:', tag)


## 5 · The table

This is the figure for the final presentation. Read 3b vs 3c for Novelty 1, and
3b vs 3a for the sampler half of Novelty 2.


In [ ]:
import json, pandas as pd

rows = []
for f in sorted(RUNS.glob('stage*_v2_seed*.json')):
    d = json.load(open(f))
    t = d.get('test') or d.get('tuned_threshold_metrics', {}).get('test', {})
    v = d.get('val')  or d.get('tuned_threshold_metrics', {}).get('val', {})
    rows.append({
        'stage': d.get('stage'), 'seed': d.get('seed'),
        'threshold': round(d.get('best_threshold', float('nan')), 4),
        'val F1':  round(v.get('f1', float('nan')), 4),
        'test P':  round(t.get('precision', float('nan')), 4),
        'test R':  round(t.get('recall', float('nan')), 4),
        'test F1': round(t.get('f1', float('nan')), 4),
        'test AUC':round(t.get('auroc', float('nan')), 4),
    })

df = pd.DataFrame(rows).sort_values(['stage', 'seed'])
print(df.to_string(index=False))
df.to_csv(RUNS / 'ablation_temporal.csv', index=False)
print('\nwrote', RUNS / 'ablation_temporal.csv')


## 6 · What to check before you trust it

- Every stage reports `protocol: temporal_snapshots_leakage_free`. If one does not,
  it did not run the pipeline you think it did.
- Test AUC should land near **0.70**, not 0.94. The old numbers were ~0.94 *because*
  of the leak — a high number here means the graph was built the old way.
- 3b's numbers should be close to the served model (precision 0.668, recall 0.271,
  F1 0.386). If they are far off, the served bundle and this run disagree and the
  bundle should be re-exported from this checkpoint.

Then download `ablation_temporal.csv` and the reports, and commit the reports to
`reports/temporal/` in the repo.


## 7 · Is the Edge-MLP broken, or is the idea wrong?

Seed 0 said Novelty 1 hurts. But the training curves say something more
specific: **every stage containing the Edge-MLP trains pathologically, and
every stage without it trains normally.**

| stage | Edge-MLP | val F1 by epoch |
|---|---|---|
| 1  | no  | 0.329 → **0.333** → 0.332 |
| 2  | yes | 0.029 → 0.058 → 0.058 (never learned) |
| 3b | yes | **0.306** → 0.295 → 0.291 (peaks at epoch 1, then declines) |
| 3c | no  | 0.339 → 0.362 → **0.364** |

There is a concrete reason to suspect the implementation. The layer differs
from the baseline in **two** ways, not one: `SAGEConv` uses `aggr='mean'`,
while `EdgeEnhancedSAGEConv` uses an unnormalised `aggr='add'`. With
in-degree up to 75, a hub's aggregated message scales with degree before
layer 2 ever sees it. Stage 2 vs 1 therefore never isolated attention.

Three one-flag remedies:

- `--attn-norm` — divide by the summed attention, giving an attention-weighted
  **mean**. Keeps relative edge importance, drops the degree scaling.
- `--attn-init-bias 3.0` — start `sigmoid` at 0.95 (pass-through) instead of
  0.48, so training begins at the baseline's behaviour rather than halving
  every message.
- `--clip-grad 1.0` — cap the first, largest gradients.

**If any of these recovers stage 2, Novelty 1 is alive.** If none do, the
negative result is precise and evidenced rather than a shrug.


In [ ]:
# Self-contained: this cell used to borrow EPOCHS from section 4,
# which you skip when only the confirmation runs are needed.
EPOCHS = 50

VARIANTS = [
    ('attnnorm',      ['--attn-norm']),
    ('initbias',      ['--attn-init-bias', '3.0']),
    ('attnnorm_init', ['--attn-norm', '--attn-init-bias', '3.0']),
    ('clip',          ['--clip-grad', '1.0']),
]

# Stage 2 is the cleanest test — it is the one that never learned at all.
# 3b matters too, since that is the configuration actually served.
for stage in ['2', '3b']:
    for name, flags in VARIANTS:
        tag = f'stage{stage}_v2_seed0_{name}'
        if (RUNS / f'{tag}.json').exists():
            print(f'skip  {tag}'); continue
        print(f'\n=== stage {stage}  {name}  {" ".join(flags)} ' + '='*30)
        rc = subprocess.call([sys.executable, 'scripts/train_temporal.py',
                              '--stage', stage, '--seed', '0',
                              '--features', 'v2', '--epochs', str(EPOCHS),
                              '--tag', name] + flags)
        if rc != 0:
            print(f'{tag} FAILED (exit {rc})'); continue
        for pat in (f'reports/temporal/{tag}.json',
                    f'checkpoints/temporal_{tag}.pt'):
            q = REPO / pat
            if q.exists():
                !cp {q} {RUNS}/
        print('saved:', tag)


### Did any of them fix it?

The bar to clear: stage 2 must beat its own broken baseline of **val F1
0.0585 / PR-AUC 0.0196**, and ideally approach stage 1's **0.3331 / 0.2475**.


In [ ]:
import json, pandas as pd

rows = []
for f in sorted(RUNS.glob('stage*_v2_seed0*.json')):
    d = json.load(open(f))
    v, t = d.get('val', {}), d.get('test', {})
    rows.append({
        'stage': d.get('stage'),
        'variant': f.stem.split('seed0')[-1].lstrip('_') or '(as published)',
        'attn_norm': d.get('attn_norm'), 'init_bias': d.get('attn_init_bias'),
        'clip': d.get('clip_grad'),
        'val F1': round(v.get('f1', float('nan')), 4),
        'val PR-AUC': round(v.get('pr_auc', float('nan')), 4),
        'test F1': round(t.get('f1', float('nan')), 4),
        'test AUC': round(t.get('auroc', float('nan')), 4),
    })
df = pd.DataFrame(rows).sort_values(['stage', 'variant'])
print(df.to_string(index=False))
df.to_csv(RUNS / 'edge_mlp_diagnosis.csv', index=False)


## 9 · Confirm the fix across seeds

`--attn-norm` recovered stage 2 outright (val F1 0.0585 → 0.2777, PR-AUC
0.0196 → 0.2073) and lifted 3b past the no-attention arm. But that is **one
seed** against 3c's three, and the margin is about two standard deviations.

Two runs settle it. The bar: **mean val F1 must clear 3c's 0.3620 ± 0.0040.**


In [ ]:
# Self-contained: this cell used to borrow EPOCHS from section 4,
# which you skip when only the confirmation runs are needed.
EPOCHS = 50

for seed in [1, 2]:
    tag = f'stage3b_v2_seed{seed}_attnnorm'
    if (RUNS / f'{tag}.json').exists():
        print('skip ', tag); continue
    print(f'\n=== stage 3b  attn-norm  seed {seed} ' + '='*34)
    rc = subprocess.call([sys.executable, 'scripts/train_temporal.py',
                          '--stage', '3b', '--seed', str(seed),
                          '--features', 'v2', '--epochs', str(EPOCHS),
                          '--attn-norm', '--tag', 'attnnorm'])
    if rc != 0:
        print(f'{tag} FAILED (exit {rc})'); continue
    for pat in (f'reports/temporal/{tag}.json',
                f'checkpoints/temporal_{tag}.pt'):
        q = REPO / pat
        if q.exists():
            !cp {q} {RUNS}/
    print('saved:', tag)


### The verdict


In [ ]:
import json, statistics as st

def seeds_of(pattern):
    out = []
    for f in sorted(RUNS.glob(pattern)):
        d = json.load(open(f))
        out.append((d['val']['f1'], d['test']['f1'], d['test']['auroc']))
    return out

fix = seeds_of('stage3b_v2_seed*_attnnorm.json')
noattn = seeds_of('stage3c_v2_seed?.json')
old = seeds_of('stage3b_v2_seed?.json')

for label, rows in (('3b + attn-norm', fix), ('3c  no attention', noattn),
                    ('3b  as served', old)):
    if not rows: continue
    v = [r[0] for r in rows]; t = [r[1] for r in rows]; a = [r[2] for r in rows]
    sd = st.stdev(v) if len(v) > 1 else 0.0
    print(f'  {label:<18} n={len(v)}  val F1 {st.mean(v):.4f} ± {sd:.4f}'
          f'   test F1 {st.mean(t):.4f}   AUC {st.mean(a):.4f}')

if len(fix) >= 3 and len(noattn) >= 3:
    d = st.mean([r[0] for r in fix]) - st.mean([r[0] for r in noattn])
    print(f'\n  attention is worth {d:+.4f} val F1 over removing it')
    print('  -> Novelty 1 CONFIRMED' if d > 0 else '  -> attention still does not help')
